# 01 — Data Ingestion & Exploratory Data Analysis
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** load the raw dataset, understand its structure, and explore it
(volume over time, category distribution, headline length, vocabulary) *before* we touch any
modelling.

**Run this in Google Colab.**

> **Important — storage:** Colab gives each notebook session a fresh, temporary virtual machine.
> Anything saved to a plain relative path (like `../data/`) disappears once the session ends or
> a different notebook is opened. So this notebook mounts your **Google Drive** and saves all
> outputs there instead — that's what lets notebook 02 (in a separate session) pick up exactly
> where this one left off. The notebook files themselves still live in GitHub; only data/outputs
> live in Drive.


In [ ]:
# --- Mount Google Drive (persistent storage across notebooks/sessions) ---
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/topic-modelling-capstone'
DATA_RAW = f'{BASE_DIR}/data/raw'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
OUTPUTS_FIGURES = f'{BASE_DIR}/outputs/figures'
OUTPUTS_MODELS = f'{BASE_DIR}/outputs/models'

for d in [DATA_RAW, DATA_PROCESSED, OUTPUTS_FIGURES, OUTPUTS_MODELS]:
    os.makedirs(d, exist_ok=True)

print("Google Drive mounted. Working directory:", BASE_DIR)


In [ ]:
# --- Setup ---
!pip install -q kagglehub pandas matplotlib seaborn wordcloud

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Get the dataset

Two options — set `USE_EXISTING_DRIVE_FILE` below accordingly:
- **`True`** — you already have the CSV in your Drive (e.g. previously downloaded manually).
  Just point `DRIVE_CSV_PATH` at it.
- **`False`** — download fresh via `kagglehub` (one-time browser login).

Using the file already in Drive is fine and reproducible as long as everyone on the team points
at the same file — just make sure `DRIVE_CSV_PATH` below matches where it actually sits in your
Drive (check via the Colab file browser sidebar, or Drive's "Copy path" option).


In [ ]:
# --- Option A: use a CSV you've already got in Drive ---
# --- Option B: download fresh via kagglehub ---
USE_EXISTING_DRIVE_FILE = True   # set to False to download via kagglehub instead

# EDIT THIS to the actual path of your file in Drive (check the Colab file browser on the left,
# or right-click the file in Drive's web UI to see its folder path)
DRIVE_CSV_PATH = '/content/drive/MyDrive/india-news-headlines.csv'

if USE_EXISTING_DRIVE_FILE:
    csv_path = DRIVE_CSV_PATH
    assert os.path.exists(csv_path), (
        f"Couldn't find a file at {csv_path} — double check the path in the Colab file browser "
        f"(folder icon on the left sidebar) and update DRIVE_CSV_PATH above."
    )
    print("Using existing file in Drive:", csv_path)
else:
    path = kagglehub.dataset_download("therohk/india-headlines-news-dataset")
    print("Dataset downloaded to:", path)
    print(os.listdir(path))
    csv_path = os.path.join(path, "india-news-headlines.csv")


In [ ]:
df = pd.read_csv(csv_path)
print(f"Rows: {len(df):,} | Columns: {list(df.columns)}")
df.head()


## 2. First look — schema, missing values, duplicates

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isna().sum())
print(f"\nExact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate headline_text (across dates): {df.duplicated(subset='headline_text').sum():,}")


In [ ]:
df['publish_date'] = pd.to_datetime(df['publish_date'], format='%Y%m%d')
df['year'] = df['publish_date'].dt.year
df['month'] = df['publish_date'].dt.month

df[['publish_date', 'year', 'month']].describe()


## 3. Volume of headlines over time

In [ ]:
yearly_counts = df.groupby('year').size()

fig, ax = plt.subplots()
yearly_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of headlines per year')
ax.set_xlabel('Year')
ax.set_ylabel('Headline count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/headlines_per_year.png', dpi=150)
plt.show()

print(yearly_counts.describe())


## 4. Category distribution

In [ ]:
df['category_top'] = df['headline_category'].astype(str).str.split('.').str[0]

top_categories = df['category_top'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_categories.plot(kind='barh', ax=ax, color='darkorange')
ax.invert_yaxis()
ax.set_title('Top 20 top-level categories by headline count')
ax.set_xlabel('Headline count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/top_categories.png', dpi=150)
plt.show()

print(f"Number of distinct top-level categories: {df['category_top'].nunique()}")
print(f"Number of distinct full categories: {df['headline_category'].nunique()}")


## 5. Headline length distribution

In [ ]:
df['headline_word_count'] = df['headline_text'].astype(str).str.split().apply(len)

fig, ax = plt.subplots()
sns.histplot(df['headline_word_count'], bins=30, ax=ax, color='seagreen')
ax.set_title('Distribution of headline length (in words)')
ax.set_xlabel('Word count')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/headline_length_dist.png', dpi=150)
plt.show()

df['headline_word_count'].describe()


In [ ]:
print("Shortest headlines:")
print(df.nsmallest(5, 'headline_word_count')[['headline_text', 'headline_word_count']])
print("\nLongest headlines:")
print(df.nlargest(5, 'headline_word_count')[['headline_text', 'headline_word_count']])


## 6. Quick vocabulary / word-frequency glance

In [ ]:
from wordcloud import WordCloud, STOPWORDS

sample_text = " ".join(df['headline_text'].astype(str).sample(200_000, random_state=RANDOM_STATE))
wc = WordCloud(width=1200, height=600, background_color='white',
                stopwords=STOPWORDS, max_words=150).generate(sample_text)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word cloud — 200k sampled headlines (raw, unprocessed)')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/wordcloud_raw_sample.png', dpi=150)
plt.show()


## 7. Build the stratified working sample

Stratified by year so every year 2001–2023 is proportionally represented. Saved to **Google
Drive**, not the local Colab disk, so notebook 02 (a separate session) can load it directly.


In [ ]:
SAMPLE_SIZE = 300_000  # ~8% of full dataset; adjust based on your time budget

sample_df = (
    df.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(frac=SAMPLE_SIZE / len(df), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)

print(f"Full dataset: {len(df):,} rows")
print(f"Sampled dataset: {len(sample_df):,} rows")
print("\nSample year distribution (should mirror full dataset proportions):")
print((sample_df['year'].value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
sample_df.to_csv(f'{DATA_PROCESSED}/headlines_sample_300k.csv', index=False)
print(f"Saved to {DATA_PROCESSED}/headlines_sample_300k.csv")
print("This file now persists in your Google Drive — notebook 02 will load it from there.")


## 8. Summary of EDA findings

*(Fill this in with the actual numbers once you run the notebook — lift this almost directly
into the report's "Understanding the Dataset" section.)*

- Total headlines: `<fill in>` across `<fill in>` years (2001–2023)
- Volume trend over time: `<fill in>`
- Dominant categories: `<fill in>`
- Typical headline length: `<fill in>`
- Missing values / duplicates: `<fill in>`
- Implication for modelling: headlines are short documents, which motivates using an
  embedding-based approach (BERTopic) over classical bag-of-words methods (LDA), which struggle
  on short documents due to sparse word co-occurrence.
